# Camada de consumo do Power BI

As consultas do `ETL CSV` mostram o resultado na tela e usam uma TEMP VIEW, que
só existe dentro da sessão do notebook. O Power BI abre a própria sessão contra
o warehouse, então não encontra nada disso.

Aqui os mesmos resultados viram objetos salvos: uma view e cinco tabelas, 239
linhas no total, que é o que o dashboard importa em vez das 172.765 da tabela
bruta.

A primeira célula carrega o CSV e cria a `tabela`, então esse notebook roda
sozinho, de cima para baixo.

In [0]:
import re
from pathlib import Path

# Lê o CSV do repositório e grava como tabela Delta. Roda uma vez: depois
# disso a `tabela` fica salva e os outros notebooks leem direto dela.

CAMINHO = str(Path('../data/DataCoSupplyChainDataset.csv').resolve())

tabela = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "ISO-8859-1")
    .csv(CAMINHO)
)

# Delta não aceita espaço, parêntese nem vírgula no nome da coluna.
# Mantemos as 53 colunas: as consultas usam 15 e filtrar aqui quebraria as
# que precisam das outras.

tabela = tabela.toDF(*[re.sub(r'[^0-9a-zA-Z_]+', '_', c).strip('_') for c in tabela.columns])

tabela.write.format("delta").mode("overwrite").saveAsTable("tabela")

print(f"{tabela.count()} linhas gravadas em `tabela`")   # esperado: 180519

In [0]:
%sql
CREATE OR REPLACE VIEW vw_enviados AS
SELECT
    Shipping_Mode, Delivery_Status, Order_Status, Order_Region,
    Customer_State, Customer_City, Market, Customer_Segment, Category_Name,
    Type AS tipo_pagamento, Order_Item_Quantity, Sales, Order_Item_Id,
    Days_for_shipping_real      AS dias_real,
    Days_for_shipment_scheduled AS dias_prog,
    Days_for_shipping_real - Days_for_shipment_scheduled AS diferenca,
    CASE WHEN Delivery_Status = 'Late delivery' THEN 1 ELSE 0 END AS atrasado,
    TO_TIMESTAMP(order_date_DateOrders, 'M/d/yyyy H:mm') AS data_pedido,
    DATE_TRUNC('MONTH', TO_TIMESTAMP(order_date_DateOrders, 'M/d/yyyy H:mm')) AS ano_mes,
    DATE_FORMAT(TO_TIMESTAMP(order_date_DateOrders, 'M/d/yyyy H:mm'), 'EEEE') AS dia_semana
FROM tabela
WHERE Delivery_Status <> 'Shipping canceled'

In [0]:
%sql
CREATE OR REPLACE TABLE bi_geral AS
SELECT COUNT(*) AS pedidos, SUM(atrasado) AS atrasados,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct,
       ROUND(AVG(dias_real), 2) AS dias_reais_medio,
       ROUND(AVG(dias_prog), 2) AS dias_prometidos_medio
FROM vw_enviados

In [0]:
%sql
CREATE OR REPLACE TABLE bi_modo_envio AS
SELECT Shipping_Mode AS modo, COUNT(*) AS pedidos, SUM(atrasado) AS atrasados,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct,
       MAX(dias_prog) AS prazo_prometido,
       ROUND(AVG(dias_real), 2) AS prazo_real_medio,
       ROUND(AVG(diferenca), 2) AS atraso_medio_dias,
       MIN(diferenca) AS menor_diferenca, MAX(diferenca) AS maior_diferenca
FROM vw_enviados GROUP BY Shipping_Mode ORDER BY taxa_atraso_pct DESC

In [0]:
%sql
CREATE OR REPLACE TABLE bi_quebras AS
SELECT 'Região' AS dimensao, Order_Region AS valor, COUNT(*) AS pedidos, SUM(atrasado) AS atrasados, ROUND(100.0*AVG(atrasado),1) AS taxa_atraso_pct FROM vw_enviados GROUP BY Order_Region HAVING COUNT(*) >= 300
UNION ALL
SELECT 'Estado do cliente', Customer_State, COUNT(*), SUM(atrasado), ROUND(100.0*AVG(atrasado),1) FROM vw_enviados GROUP BY Customer_State HAVING COUNT(*) >= 300
UNION ALL
SELECT 'Cidade do cliente', Customer_City, COUNT(*), SUM(atrasado), ROUND(100.0*AVG(atrasado),1) FROM vw_enviados GROUP BY Customer_City HAVING COUNT(*) >= 300
UNION ALL
SELECT 'Mercado', Market, COUNT(*), SUM(atrasado), ROUND(100.0*AVG(atrasado),1) FROM vw_enviados GROUP BY Market HAVING COUNT(*) >= 300
UNION ALL
SELECT 'Categoria', Category_Name, COUNT(*), SUM(atrasado), ROUND(100.0*AVG(atrasado),1) FROM vw_enviados GROUP BY Category_Name HAVING COUNT(*) >= 300
UNION ALL
SELECT 'Segmento', Customer_Segment, COUNT(*), SUM(atrasado), ROUND(100.0*AVG(atrasado),1) FROM vw_enviados GROUP BY Customer_Segment HAVING COUNT(*) >= 300
UNION ALL
SELECT 'Tipo de pagamento', tipo_pagamento, COUNT(*), SUM(atrasado), ROUND(100.0*AVG(atrasado),1) FROM vw_enviados GROUP BY tipo_pagamento HAVING COUNT(*) >= 300
UNION ALL
SELECT 'Status do pedido', Order_Status, COUNT(*), SUM(atrasado), ROUND(100.0*AVG(atrasado),1) FROM vw_enviados GROUP BY Order_Status HAVING COUNT(*) >= 300
UNION ALL
SELECT 'Qtd. de itens', CAST(Order_Item_Quantity AS STRING), COUNT(*), SUM(atrasado), ROUND(100.0*AVG(atrasado),1) FROM vw_enviados GROUP BY Order_Item_Quantity HAVING COUNT(*) >= 300
UNION ALL
SELECT 'Dia da semana', dia_semana, COUNT(*), SUM(atrasado), ROUND(100.0*AVG(atrasado),1) FROM vw_enviados GROUP BY dia_semana HAVING COUNT(*) >= 300

In [0]:
%sql
CREATE OR REPLACE TABLE bi_serie_mensal AS
SELECT ano_mes, COUNT(*) AS pedidos, SUM(atrasado) AS atrasados,
       ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM vw_enviados GROUP BY ano_mes ORDER BY ano_mes

In [0]:
%sql
CREATE OR REPLACE TABLE bi_faixa_valor AS
SELECT quartil, ROUND(MIN(Sales), 2) AS valor_min, ROUND(MAX(Sales), 2) AS valor_max,
       COUNT(*) AS pedidos, ROUND(100.0 * AVG(atrasado), 1) AS taxa_atraso_pct
FROM (SELECT atrasado, Sales,
             NTILE(4) OVER (ORDER BY Sales, Order_Item_Id) AS quartil
      FROM vw_enviados)
GROUP BY quartil ORDER BY quartil